In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

# opt = torch_numopt.ConjugateGradient(model=model, lr=1e-4, line_search_method="const", cg_method="PR")
opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="interpolate", line_search_cond="armijo", cg_method="PR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="wolfe", cg_method="PR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="strong-wolfe", cg_method="PR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="goldstein", cg_method="PR")


model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.0903245136141777
epoch:  1, loss: 0.04392770677804947
epoch:  2, loss: 0.03573697805404663
epoch:  3, loss: 0.03487108275294304
epoch:  4, loss: 0.034805770963430405
epoch:  5, loss: 0.03473988175392151
epoch:  6, loss: 0.0346742607653141
epoch:  7, loss: 0.034596655517816544
epoch:  8, loss: 0.03451671078801155
epoch:  9, loss: 0.034389954060316086
epoch:  10, loss: 0.03424815833568573
epoch:  11, loss: 0.033032115548849106
epoch:  12, loss: 0.03278982639312744
epoch:  13, loss: 0.031229421496391296
epoch:  14, loss: 0.03016897849738598
epoch:  15, loss: 0.02984997257590294
epoch:  16, loss: 0.02900947444140911
epoch:  17, loss: 0.028539670631289482
epoch:  18, loss: 0.0282734427601099
epoch:  19, loss: 0.022280806675553322
epoch:  20, loss: 0.021626705303788185
epoch:  21, loss: 0.021237874403595924
epoch:  22, loss: 0.020943012088537216
epoch:  23, loss: 0.017346756532788277
epoch:  24, loss: 0.01648220419883728
epoch:  25, loss: 0.015607991255819798
epoch:  26, l

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7815593989151675
Test metrics:  R2 = 0.7072262248927718


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

# opt = torch_numopt.ConjugateGradient(model=model, lr=1e-4, line_search_method="const", cg_method="FR")
opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="interpolate", line_search_cond="armijo", cg_method="FR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="wolfe", cg_method="FR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="strong-wolfe", cg_method="FR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="goldstein", cg_method="FR")

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.1249898299574852
epoch:  1, loss: 0.04814499244093895
epoch:  2, loss: 0.03694232925772667
epoch:  3, loss: 0.0358392558991909
epoch:  4, loss: 0.03580056503415108
epoch:  5, loss: 0.03576299920678139
epoch:  6, loss: 0.03572655841708183
epoch:  7, loss: 0.03569159656763077
epoch:  8, loss: 0.03565654531121254
epoch:  9, loss: 0.03562110289931297
epoch:  10, loss: 0.03558582812547684
epoch:  11, loss: 0.03554948791861534
epoch:  12, loss: 0.035513561218976974
epoch:  13, loss: 0.03547631576657295
epoch:  14, loss: 0.03543924167752266
epoch:  15, loss: 0.03539953753352165
epoch:  16, loss: 0.035358525812625885
epoch:  17, loss: 0.03531390056014061
epoch:  18, loss: 0.03526747226715088
epoch:  19, loss: 0.03521224111318588
epoch:  20, loss: 0.035155151039361954
epoch:  21, loss: 0.035070110112428665
epoch:  22, loss: 0.03498045355081558
epoch:  23, loss: 0.03465801104903221
epoch:  24, loss: 0.03433606028556824
epoch:  25, loss: 0.03419329971075058
epoch:  26, loss: 0.

In [10]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7454465398386013
Test metrics:  R2 = 0.6716967163499958
